In [1]:
!pip install datasets==2.19.0 transformers timm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
import torch
print(torch.cuda.is_available())   # Треба: True
print(torch.cuda.get_device_name(0))  # Треба: Tesla T4

True
Tesla T4


In [3]:
from datasets import load_dataset

print("Вчитувам датасет...")
dataset = load_dataset("tanganke/stanford_cars")
print(dataset)

Вчитувам датасет...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8144 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating contrast split:   0%|          | 0/8041 [00:00<?, ? examples/s]

...

Generating jpeg_compression split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating motion_blur split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating pixelate split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating spatter split:   0%|          | 0/8041 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 8144
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    contrast: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    gaussian_noise: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    impulse_noise: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    jpeg_compression: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    motion_blur: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    pixelate: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
    spatter: Dataset({
        features: ['image', 'label'],
        num_rows: 8041
    })
})


In [5]:
from transformers import ViTForImageClassification, AutoImageProcessor

model_name = "google/vit-base-patch16-224"

image_processor = AutoImageProcessor.from_pretrained(model_name, use_fast=True)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=196,
    ignore_mismatched_sizes=True
)

print("Моделот е подготвен!")
print(f"Број на параметри: {sum(p.numel() for p in model.parameters()):,}")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([196, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([196])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Моделот е подготвен!
Број на параметри: 85,949,380


In [12]:
from transformers import AutoImageProcessor
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean,
                         std=image_processor.image_std)
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean,
                         std=image_processor.image_std)
])

class CarsDataset(Dataset):
    def __init__(self, hf_data, transform):
        self.data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]["image"].convert("RGB")
        label = self.data[idx]["label"]
        return {"pixel_values": self.transform(image), "labels": label}

train_dataset = CarsDataset(dataset["train"], train_transforms)
test_dataset  = CarsDataset(dataset["test"],  test_transforms)

print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")

Train: 8144 | Test: 8041


In [13]:
import torch
from torch.utils.data import DataLoader

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels = torch.tensor([b["labels"] for b in batch])
    return {"pixel_values": pixel_values, "labels": labels}

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Train батчеви: {len(train_loader)} | Test батчеви: {len(test_loader)}")

Train батчеви: 255 | Test батчеви: 252


In [14]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Модел на: {device}")

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

print("Optimizer и scheduler подготвени!")

Модел на: cuda
Optimizer и scheduler подготвени!


In [15]:
from tqdm import tqdm
import os

os.makedirs("checkpoints", exist_ok=True)  # Прави папка за зачувување

EPOCHS = 10
best_accuracy = 0.0

for epoch in range(EPOCHS):
    # ТРЕНИНГ
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_acc  = correct / total * 100
    train_loss = total_loss / len(train_loader)

    # ЕВАЛУАЦИЈА
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Eval]"):
            pixel_values = batch["pixel_values"].to(device)
            labels       = batch["labels"].to(device)
            outputs      = model(pixel_values=pixel_values)
            preds        = outputs.logits.argmax(dim=-1)
            correct     += (preds == labels).sum().item()
            total       += labels.size(0)

    test_acc = correct / total * 100
    scheduler.step()

    print(f"\nEpoch {epoch+1}: Loss={train_loss:.4f} | Train={train_acc:.2f}% | Test={test_acc:.2f}%")

    # Зачувај СЕКОЈА епоха
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_acc": train_acc,
            "test_acc": test_acc,
            "loss": train_loss,
        },
        f"checkpoints/vit_epoch_{epoch+1}_acc{test_acc:.2f}.pt"
    )
    print(f" Epoch {epoch+1} зачувана!")

    # Зачувај и посебно најдобриот
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save(model.state_dict(), "vit_stanford_cars_best.pt")
        print(f" Најдобар модел! ({test_acc:.2f}%)")

Epoch 1/10 [Eval]: 100%|██████████| 252/252 [01:51<00:00,  2.26it/s]



Epoch 1: Loss=4.9527 | Train=4.60% | Test=11.62%
 Epoch 1 зачувана!
 Најдобар модел! (11.62%)


Epoch 2/10 [Eval]: 100%|██████████| 252/252 [01:52<00:00,  2.24it/s]



Epoch 2: Loss=3.9221 | Train=24.67% | Test=29.25%
 Epoch 2 зачувана!
 Најдобар модел! (29.25%)


Epoch 3/10 [Eval]: 100%|██████████| 252/252 [01:52<00:00,  2.23it/s]


...


Epoch 8: Loss=1.2463 | Train=88.97% | Test=69.66%
 Epoch 8 зачувана!
 Најдобар модел! (69.66%)


Epoch 9/10 [Eval]: 100%|██████████| 252/252 [01:52<00:00,  2.23it/s]



Epoch 9: Loss=1.1709 | Train=90.48% | Test=70.66%
 Epoch 9 зачувана!
 Најдобар модел! (70.66%)


Epoch 10/10 [Eval]: 100%|██████████| 252/252 [01:52<00:00,  2.24it/s]



Epoch 10: Loss=1.1304 | Train=91.32% | Test=70.82%
 Epoch 10 зачувана!
 Најдобар модел! (70.82%)


In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import top_k_accuracy_score
import numpy as np

model.eval()
all_preds  = []
all_labels = []
all_logits = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Евалуација"):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)
        outputs      = model(pixel_values=pixel_values)
        preds        = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_logits.extend(outputs.logits.cpu().numpy())

all_logits = np.array(all_logits)
top1 = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
top5 = top_k_accuracy_score(all_labels, all_logits, k=5) * 100

print(f"Top-1 Accuracy: {top1:.2f}%")
print(f"Top-5 Accuracy: {top5:.2f}%")
print(classification_report(all_labels, all_preds, zero_division=0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    "vit_stanford_cars_best.pt",
    "/content/drive/MyDrive/vit_stanford_cars_best.pt"
)
print("Моделот е зачуван на Google Drive!")

In [ ]:
import pandas as pd

results = {
    "Model":          ["EfficientNet-B0 v1", "EfficientNet-B0 v2", "ConvNeXt-Tiny", "ViT-B/16"],
    "Top-1 Accuracy": ["83.08%",             "81.64%",             "87.08%",         f"{top1:.2f}%"],
    "Top-5 Accuracy": ["95.20%",             "94.50%",             "96.81%",         f"{top5:.2f}%"],
    "Model Size":     ["17 MB",              "17 MB",              "112 MB",         "330 MB"],
}

df = pd.DataFrame(results)
print(df.to_string(index=False))